# Notebook 01: Memory Stream - ระบบความทรงจำของ Generative Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tiraphap/Gen-Agent-Lectures-at-Econ-TU/blob/master/teaching/notebooks/01_memory_stream.ipynb)

## สิ่งที่จะได้เรียนรู้
1. **Memory Stream คืออะไร** - ระบบความทรงจำที่เก็บทุกประสบการณ์ของ agent
2. **สูตร Retrieval** - วิธีค้นหาความทรงจำที่เกี่ยวข้อง: `score = recency × importance × relevance`
3. **Hands-on** - สร้าง Memory Stream จาก scratch และทดลองค้นหา

**ไม่ต้องใช้ LLM / API Key** - ทุกอย่างรันได้เลย!

---

## Background: ทำไมต้องมี Memory?

ลองนึกภาพว่าคุณเป็น AI agent ที่ต้องตัดสินใจว่าจะไปซื้อของที่ไหน:

- **ถ้าไม่มี memory**: ทุกวันเหมือนเกิดใหม่ ไม่รู้ว่าร้านไหนดี ร้านไหนแย่
- **ถ้ามี memory**: จำได้ว่าเมื่อวาน Big C มีโปรลดนม 30% → ตัดสินใจไปซื้อได้ดีขึ้น

### จาก Paper: "Generative Agents" (Stanford, 2023)

Memory Stream เป็นหัวใจของ Generative Agent:
- เก็บทุกอย่างที่ agent **สังเกตเห็น** (observations)
- เก็บ **ข้อคิด** ที่ agent สรุปจากประสบการณ์ (reflections)
- เก็บ **แผนการ** ที่ agent วางไว้ (plans)

ปัญหาคือ: ถ้ามี memory เป็นพันๆ อัน จะเลือกอันไหนมาใช้ตัดสินใจ?

**คำตอบ: สูตร Retrieval!**

---
**ผศ.ดร.ถิรภาพ ฟักทอง** | คณะเศรษฐศาสตร์ มหาวิทยาลัยธรรมศาสตร์

## Part 1: สร้าง Memory class

แต่ละ memory จะเก็บข้อมูลเหล่านี้:
- `content`: เนื้อหาของความทรงจำ (เป็นภาษาธรรมชาติ)
- `created_at`: เวลาที่เกิดเหตุการณ์
- `importance`: ความสำคัญ (1-10) - ปกติ LLM จะเป็นคนให้คะแนน
- `memory_type`: ประเภท (observation / reflection / plan)

In [ ]:
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import Optional
import math


@dataclass
class Memory:
    """
    ความทรงจำหนึ่งอัน ของ Generative Agent
    
    ตัวอย่าง:
        Memory(
            content="ไป Big C เห็นนมลดราคา 30%",
            created_at=datetime(2024, 1, 13, 10, 0),
            importance=7,
            memory_type="observation"
        )
    """
    content: str                          # เนื้อหาความทรงจำ
    created_at: datetime                  # เวลาที่เกิด
    importance: int                       # ความสำคัญ 1-10
    memory_type: str = "observation"      # observation / reflection / plan
    related_store: Optional[str] = None   # ร้านค้าที่เกี่ยวข้อง (ถ้ามี)
    access_count: int = 0                 # จำนวนครั้งที่ถูกเรียกใช้

    def __repr__(self):
        return f"Memory('{self.content[:40]}...', imp={self.importance}, type={self.memory_type})"


print("Memory class สร้างเสร็จแล้ว!")
print()

# ทดลองสร้าง memory 1 อัน
test_memory = Memory(
    content="ไป Big C เห็นนมลดราคา 30% น่าสนใจมาก",
    created_at=datetime(2024, 1, 13, 10, 0),
    importance=7,
    memory_type="observation",
    related_store="Big C"
)
print(f"ตัวอย่าง Memory: {test_memory}")
print(f"  เนื้อหา: {test_memory.content}")
print(f"  ความสำคัญ: {test_memory.importance}/10")
print(f"  ประเภท: {test_memory.memory_type}")

## Part 2: สูตร Retrieval - หัวใจของระบบ Memory

เมื่อ agent ต้องการค้นหาความทรงจำที่เกี่ยวข้อง จะใช้สูตร:

### `score = recency × importance × relevance`

ประกอบด้วย 3 ส่วน:

| ส่วน | สูตร | ความหมาย |
|------|------|----------|
| **Recency** (ความใหม่) | `0.99 ^ hours_since_event` | ยิ่งเก่า ยิ่งได้คะแนนน้อย |
| **Importance** (ความสำคัญ) | `importance / 10` | คะแนน 1-10 ที่ LLM ให้ |
| **Relevance** (ความเกี่ยวข้อง) | `keyword_overlap(query, memory)` | ตรงกับสิ่งที่กำลังค้นหาแค่ไหน |

### ทำไมต้องคูณกัน?

- ถ้า recency สูง แต่ relevance = 0 → score = 0 (เพิ่งเกิด แต่ไม่เกี่ยว → ไม่เอา)
- ถ้า relevance สูง แต่ recency ต่ำมาก → score ต่ำ (เกี่ยวข้อง แต่เก่ามาก → ลดความสำคัญ)
- ทั้ง 3 ส่วนต้องสูงพร้อมกัน → ได้ score สูง

In [ ]:
# ============================================================
# Part 2a: Recency Score - ความใหม่ของความทรงจำ
# ============================================================
#
# สูตร: recency = 0.99 ^ hours_since_event
#
# หลักการ: ความทรงจำที่เก่ากว่าจะค่อยๆ จางลง (exponential decay)
# - 0 ชั่วโมงที่แล้ว → 0.99^0 = 1.000 (จำได้ชัดมาก)
# - 1 ชั่วโมงที่แล้ว → 0.99^1 = 0.990
# - 24 ชั่วโมง (1 วัน) → 0.99^24 = 0.786
# - 168 ชั่วโมง (1 สัปดาห์) → 0.99^168 = 0.186
# - 720 ชั่วโมง (1 เดือน) → 0.99^720 = 0.001 (แทบจำไม่ได้)

RECENCY_DECAY = 0.99  # ค่าคงที่สำหรับ exponential decay

def calculate_recency(memory: Memory, current_time: datetime) -> float:
    """
    คำนวณ recency score
    
    ยิ่ง memory เก่า → score ยิ่งต่ำ
    ใช้ exponential decay: 0.99^hours
    """
    hours_elapsed = (current_time - memory.created_at).total_seconds() / 3600
    return RECENCY_DECAY ** hours_elapsed


# ทดลอง: ดูว่า recency ลดลงอย่างไรตามเวลา
print("=" * 60)
print("Recency Score: ยิ่งเก่า ยิ่งจางลง")
print("=" * 60)
print(f"{'เวลาที่ผ่านไป':<25} {'Recency Score':<15} {'กราฟ'}")
print("-" * 60)

now = datetime(2024, 1, 14, 10, 0)  # เวลาปัจจุบัน

time_points = [
    (0, "เพิ่งเกิดขึ้น"),
    (1, "1 ชั่วโมงที่แล้ว"),
    (6, "6 ชั่วโมงที่แล้ว"),
    (24, "1 วันที่แล้ว"),
    (48, "2 วันที่แล้ว"),
    (168, "1 สัปดาห์ที่แล้ว"),
    (336, "2 สัปดาห์ที่แล้ว"),
    (720, "1 เดือนที่แล้ว"),
]

for hours, label in time_points:
    mem = Memory(content="test", created_at=now - timedelta(hours=hours), importance=5)
    score = calculate_recency(mem, now)
    bar = "█" * int(score * 30)
    print(f"  {label:<23} {score:<15.4f} {bar}")

print()
print("สังเกต: ความทรงจำที่เก่ากว่า 1 เดือน แทบจะหายไปเลย (score ≈ 0)")

In [ ]:
# ============================================================
# Part 2b: Importance Score - ความสำคัญของความทรงจำ
# ============================================================
#
# ใน paper จริง: LLM จะเป็นคนให้คะแนน 1-10
# เราจะ normalize เป็น 0-1 โดยหาร 10
#
# ตัวอย่างคะแนน:
# - "เดินผ่านร้าน 7-11" → importance = 2 (เรื่องธรรมดา)
# - "เห็นโปรโมชั่นนมลด 30%" → importance = 6 (น่าสนใจ)
# - "ซื้อของไม่ทัน ของหมดก่อน!" → importance = 8 (ประสบการณ์สำคัญ)
# - "ร้านประจำปิดถาวร" → importance = 10 (เปลี่ยนพฤติกรรมเลย)

def calculate_importance(memory: Memory) -> float:
    """
    Normalize importance score จาก 1-10 เป็น 0.1-1.0
    """
    return memory.importance / 10.0


# ตัวอย่าง importance levels
print("=" * 60)
print("Importance Score: LLM เป็นคนให้คะแนน 1-10")
print("=" * 60)

examples = [
    ("เดินผ่านร้าน 7-11", 2, "เรื่องธรรมดามาก"),
    ("ซื้อกาแฟตอนเช้า", 3, "กิจวัตรปกติ"),
    ("เห็นโปรนมลด 30% ที่ Big C", 6, "น่าสนใจ อาจเปลี่ยนแผน"),
    ("พนักงานร้านบริการดีมาก", 7, "ส่งผลต่อการกลับมาซื้อ"),
    ("ของที่ต้องการหมด! ไม่ทันซื้อ", 8, "ประสบการณ์แย่ จำนาน"),
    ("ร้าน CJ More ปิดปรับปรุง", 10, "เปลี่ยนพฤติกรรมทั้งหมด"),
]

print(f"{'ความทรงจำ':<35} {'คะแนน':<8} {'Normalized':<12} {'อธิบาย'}")
print("-" * 80)
for content, imp, desc in examples:
    norm = imp / 10.0
    bar = "█" * int(norm * 20)
    print(f"  {content:<33} {imp}/10    {norm:<12.1f} {bar} {desc}")

In [ ]:
# ============================================================
# Part 2c: Relevance Score - ความเกี่ยวข้องกับ query
# ============================================================
#
# Phase 2 (ปัจจุบัน): ใช้ keyword matching (จับคู่คำ)
# Phase 3+ (อนาคต): ใช้ vector embeddings + cosine similarity
#
# วิธี keyword matching:
# 1. แยกคำใน query → เช่น "นม โปรโมชั่น" → {"นม", "โปรโมชั่น"}
# 2. แยกคำใน memory → เช่น "ไป Big C เห็นนมลด" → {"ไป", "big", "c", "เห็น", "นม", "ลด"}
# 3. นับคำที่ตรงกัน → "นม" ตรง 1 คำ จาก 2 คำใน query → 0.5

def calculate_relevance(memory: Memory, query: str) -> float:
    """
    คำนวณ relevance score จาก keyword matching
    
    วิธี: นับจำนวนคำที่ตรงกันระหว่าง query กับ memory
    แล้วหารด้วยจำนวนคำทั้งหมดใน query
    
    Return: float 0.0 - 1.0
    """
    # แยกคำ (split by space) แล้วทำเป็น set (ไม่ซ้ำ)
    query_words = set(query.lower().split())
    memory_words = set(memory.content.lower().split())
    
    if not query_words:
        return 0.0
    
    # นับคำที่ตรงกัน (intersection)
    matching_words = query_words & memory_words  # & = intersection
    
    # คำนวณ score
    score = len(matching_words) / len(query_words)
    return min(1.0, score)  # cap ที่ 1.0


# ทดลอง relevance scoring
print("=" * 70)
print("Relevance Score: ความเกี่ยวข้องกับสิ่งที่ค้นหา")
print("=" * 70)

test_memories = [
    "ไป Big C เห็นนมลดราคา 30% น่าสนใจมาก",
    "ซื้อนม 3 กล่องที่ Lotus's เมื่อวาน",
    "พนักงาน Lotus's บริการดีมาก ช่วยหาของ",
    "วันนี้ฝนตก ไม่อยากออกจากบ้าน",
    "ลูกชอบกินนมยี่ห้อ Dutch Mill มาก",
]

query = "นม โปรโมชั่น ลดราคา"
print(f"\nQuery: \"{query}\"")
print(f"คำใน query: {set(query.lower().split())}")
print("-" * 70)

for content in test_memories:
    mem = Memory(content=content, created_at=datetime.now(), importance=5)
    score = calculate_relevance(mem, query)
    
    # แสดงคำที่ตรงกัน
    q_words = set(query.lower().split())
    m_words = set(content.lower().split())
    matches = q_words & m_words
    
    bar = "█" * int(score * 20)
    match_str = ", ".join(matches) if matches else "ไม่มี"
    print(f"  Memory: \"{content}\"")
    print(f"    คำที่ตรง: [{match_str}]  Score: {score:.2f}  {bar}")
    print()

## Part 3: รวมทุกอย่าง - สร้าง MemoryStream class

ตอนนี้เรามีทั้ง 3 ส่วนแล้ว ก็ถึงเวลารวมเข้าด้วยกัน:

```
score = recency × importance × relevance
```

MemoryStream จะเก็บ memories ทั้งหมดของ agent คนหนึ่ง
และมี method `retrieve()` สำหรับค้นหา memories ที่เกี่ยวข้องที่สุด

In [ ]:
class MemoryStream:
    """
    ระบบความทรงจำของ Generative Agent
    
    เก็บ memories ทั้งหมดของ agent คนหนึ่ง
    และค้นหาด้วยสูตร: score = recency × importance × relevance
    
    จาก Paper: "Generative Agents" (Stanford, 2023)
    """
    
    RECENCY_DECAY = 0.99  # ค่า decay ต่อชั่วโมง
    
    def __init__(self, agent_name: str):
        """สร้าง MemoryStream ใหม่สำหรับ agent"""
        self.agent_name = agent_name
        self.memories: list[Memory] = []
    
    def add(self, content: str, importance: int, memory_type: str = "observation",
            created_at: datetime = None, related_store: str = None) -> Memory:
        """
        เพิ่ม memory ใหม่เข้า stream
        
        Args:
            content: เนื้อหาความทรงจำ
            importance: ความสำคัญ 1-10
            memory_type: "observation", "reflection", หรือ "plan"
            created_at: เวลาที่เกิด (default = ตอนนี้)
            related_store: ร้านค้าที่เกี่ยวข้อง
        """
        memory = Memory(
            content=content,
            created_at=created_at or datetime.now(),
            importance=importance,
            memory_type=memory_type,
            related_store=related_store
        )
        self.memories.append(memory)
        return memory
    
    def _calc_recency(self, memory: Memory, current_time: datetime) -> float:
        """คำนวณ recency: 0.99^hours"""
        hours = (current_time - memory.created_at).total_seconds() / 3600
        return self.RECENCY_DECAY ** hours
    
    def _calc_relevance(self, memory: Memory, query: str) -> float:
        """คำนวณ relevance: keyword matching"""
        q_words = set(query.lower().split())
        m_words = set(memory.content.lower().split())
        if not q_words:
            return 0.0
        return min(1.0, len(q_words & m_words) / len(q_words))
    
    def retrieve(self, query: str, top_k: int = 5,
                 current_time: datetime = None) -> list[tuple[Memory, dict]]:
        """
        ค้นหา memories ที่เกี่ยวข้องที่สุด
        
        Args:
            query: สิ่งที่กำลังค้นหา (เช่น "นม โปรโมชั่น")
            top_k: จำนวนผลลัพธ์สูงสุด
            current_time: เวลาปัจจุบัน
        
        Returns:
            list of (Memory, score_breakdown_dict)
            score_breakdown มี: recency, importance, relevance, total
        """
        if current_time is None:
            current_time = datetime.now()
        
        results = []
        for mem in self.memories:
            recency = self._calc_recency(mem, current_time)
            importance = mem.importance / 10.0
            relevance = self._calc_relevance(mem, query)
            total = recency * importance * relevance
            
            results.append((mem, {
                "recency": recency,
                "importance": importance,
                "relevance": relevance,
                "total": total
            }))
        
        # เรียงตาม total score จากมากไปน้อย
        results.sort(key=lambda x: x[1]["total"], reverse=True)
        return results[:top_k]
    
    def count(self) -> int:
        return len(self.memories)


print("MemoryStream class สร้างเสร็จแล้ว!")
print("พร้อมใช้งานด้วยสูตร: score = recency × importance × relevance")

## Part 4: ทดลองจริง - สมศรี กับความทรงจำ 15 อัน

มาสร้าง MemoryStream ให้ **สมศรี** (แม่บ้าน อายุ 45 ปี) กันเถอะ!

สมศรี เป็นคนดูแลครอบครัว 4 คน ชอบดูโปรโมชั่น ละเอียดเรื่องราคา

เราจะเพิ่ม memories จากช่วงเวลาต่างๆ แล้วทดลอง query ดู

In [ ]:
# สร้าง MemoryStream ให้สมศรี
somsri = MemoryStream(agent_name="สมศรี")

# เวลาปัจจุบัน: วันเสาร์ 13 ม.ค. 2024 เวลา 10:00 น.
NOW = datetime(2024, 1, 13, 10, 0)

# ========================================
# เพิ่ม Memories จากช่วงเวลาต่างๆ
# ========================================

print("กำลังเพิ่ม memories ให้สมศรี...")
print("=" * 70)

# --- 2 สัปดาห์ที่แล้ว ---
somsri.add(
    "ไป Lotus's ซื้อของ พนักงานช่วยหาของดีมาก บริการประทับใจ",
    importance=7, memory_type="observation", related_store="Lotus's",
    created_at=NOW - timedelta(days=14)
)
print("  + [2 สัปดาห์ก่อน] ประสบการณ์ดีที่ Lotus's (importance=7)")

somsri.add(
    "ซื้อของที่ Makro ราคาถูกแต่ต้องซื้อเยอะ เก็บไม่หมด",
    importance=5, memory_type="observation", related_store="Makro",
    created_at=NOW - timedelta(days=13)
)
print("  + [13 วันก่อน] ประสบการณ์ที่ Makro (importance=5)")

# --- 1 สัปดาห์ที่แล้ว ---
somsri.add(
    "Lotus's มีโปรลดนม Dutch Mill 20% ซื้อมา 2 กล่อง",
    importance=6, memory_type="observation", related_store="Lotus's",
    created_at=NOW - timedelta(days=7)
)
print("  + [1 สัปดาห์ก่อน] ซื้อนมโปรที่ Lotus's (importance=6)")

somsri.add(
    "ไป Big C วันเสาร์ คนเยอะมาก ต่อคิวนาน ไม่ชอบเลย",
    importance=7, memory_type="observation", related_store="Big C",
    created_at=NOW - timedelta(days=7, hours=2)
)
print("  + [1 สัปดาห์ก่อน] ประสบการณ์แย่ที่ Big C (importance=7)")

somsri.add(
    "Big C มีของครบกว่า Lotus's โดยเฉพาะผงซักฟอก มีหลายยี่ห้อ",
    importance=5, memory_type="reflection", related_store="Big C",
    created_at=NOW - timedelta(days=6)
)
print("  + [6 วันก่อน] [Reflection] Big C ของครบ (importance=5)")

# --- 3-5 วันที่แล้ว ---
somsri.add(
    "ทำอาหารเย็นให้ครอบครัว ใช้ไข่หมด 10 ฟอง",
    importance=4, memory_type="observation",
    created_at=NOW - timedelta(days=5)
)
print("  + [5 วันก่อน] ใช้ไข่หมด (importance=4)")

somsri.add(
    "ลูกบอกว่าอยากกินนม Dutch Mill สตรอเบอร์รี่",
    importance=5, memory_type="observation",
    created_at=NOW - timedelta(days=4)
)
print("  + [4 วันก่อน] ลูกอยากกินนม (importance=5)")

somsri.add(
    "ดูใบปลิว Big C เห็นโปรลดนม 30% สัปดาห์นี้",
    importance=7, memory_type="observation", related_store="Big C",
    created_at=NOW - timedelta(days=3)
)
print("  + [3 วันก่อน] เห็นใบปลิว Big C ลดนม 30% (importance=7)")

somsri.add(
    "ผงซักฟอกเหลือน้อยแล้ว ต้องซื้อเพิ่ม",
    importance=6, memory_type="observation",
    created_at=NOW - timedelta(days=3)
)
print("  + [3 วันก่อน] ผงซักฟอกเหลือน้อย (importance=6)")

# --- เมื่อวาน ---
somsri.add(
    "เช็คตู้เย็น นมเหลือ 1 กล่อง ต้องซื้อเพิ่มแน่ๆ",
    importance=8, memory_type="observation",
    created_at=NOW - timedelta(days=1)
)
print("  + [เมื่อวาน] นมเหลือ 1 กล่อง (importance=8)")

somsri.add(
    "ไข่หมดไปเมื่อวาน ต้องซื้อด้วย",
    importance=7, memory_type="observation",
    created_at=NOW - timedelta(days=1, hours=2)
)
print("  + [เมื่อวาน] ไข่หมด (importance=7)")

somsri.add(
    "CJ More ใกล้บ้านที่สุด เดิน 5 นาทีถึง แต่ของไม่ค่อยครบ",
    importance=5, memory_type="reflection", related_store="CJ More",
    created_at=NOW - timedelta(hours=20)
)
print("  + [20 ชม.ก่อน] [Reflection] CJ More ใกล้แต่ของไม่ครบ (importance=5)")

# --- วันนี้ ---
somsri.add(
    "วันนี้วันเสาร์ ไม่รีบ อากาศดี น่าออกไปซื้อของ",
    importance=4, memory_type="observation",
    created_at=NOW - timedelta(hours=2)
)
print("  + [2 ชม.ก่อน] วันเสาร์อากาศดี (importance=4)")

somsri.add(
    "สามีบอกว่าอยากกินส้มตำ ต้องซื้อมะละกอด้วย",
    importance=5, memory_type="observation",
    created_at=NOW - timedelta(hours=1)
)
print("  + [1 ชม.ก่อน] สามีอยากกินส้มตำ (importance=5)")

somsri.add(
    "ควรไปซื้อของวันนี้ เพราะนมกับไข่เหลือน้อย",
    importance=8, memory_type="plan",
    created_at=NOW - timedelta(minutes=30)
)
print("  + [30 นาทีก่อน] [Plan] ต้องไปซื้อของวันนี้ (importance=8)")

print(f"\nรวม: {somsri.count()} memories")
print("=" * 70)

In [ ]:
# ============================================================
# ทดลอง Query 1: "นม โปรโมชั่น ลดราคา"
# ============================================================
#
# สมศรี กำลังคิดว่า: "มีร้านไหนมีโปรนมบ้าง?"

query = "นม โปรโมชั่น ลดราคา"

print("=" * 70)
print(f"QUERY: \"{query}\"")
print(f"สมศรี กำลังคิดว่า: 'มีร้านไหนมีโปรนมบ้าง?'")
print("=" * 70)

results = somsri.retrieve(query, top_k=10, current_time=NOW)

print(f"\n{'#':<4} {'Score':<8} {'Rec.':<7} {'Imp.':<7} {'Rel.':<7} {'Type':<12} {'Memory'}")
print("-" * 90)

for i, (mem, scores) in enumerate(results, 1):
    # สร้าง visual bar
    bar_len = int(scores['total'] * 50)
    bar = "█" * bar_len
    
    # ตัด content ให้สั้นลง
    short_content = mem.content[:45] + "..." if len(mem.content) > 45 else mem.content
    
    print(f"  {i:<3} {scores['total']:<8.4f} {scores['recency']:<7.3f} {scores['importance']:<7.1f} {scores['relevance']:<7.2f} [{mem.memory_type:<10}] {short_content}")
    print(f"      {bar}")

# สรุปผล
if results:
    top_mem = results[0][0]
    print(f"\nTop memory: \"{top_mem.content}\"")
    print(f"ทำไมอันนี้ได้อันดับ 1?")
    top_scores = results[0][1]
    print(f"  - recency = {top_scores['recency']:.3f} (ยังจำได้ชัด)")
    print(f"  - importance = {top_scores['importance']:.1f} (ค่อนข้างสำคัญ)")
    print(f"  - relevance = {top_scores['relevance']:.2f} (ตรงกับ query)")
    print(f"  → total = {top_scores['recency']:.3f} × {top_scores['importance']:.1f} × {top_scores['relevance']:.2f} = {top_scores['total']:.4f}")

In [ ]:
# ============================================================
# ทดลอง Query 2: "บริการ ร้าน ดี สะดวก"
# ============================================================
#
# สมศรี กำลังคิดว่า: "ร้านไหนบริการดีบ้าง?"
# สังเกตว่า query ต่างกัน → ผลลัพธ์ต่างกัน!

query2 = "บริการ ร้าน ดี สะดวก"

print("=" * 70)
print(f"QUERY: \"{query2}\"")
print(f"สมศรี กำลังคิดว่า: 'ร้านไหนบริการดี ไปแล้วสะดวก?'")
print("=" * 70)

results2 = somsri.retrieve(query2, top_k=10, current_time=NOW)

print(f"\n{'#':<4} {'Score':<8} {'Rec.':<7} {'Imp.':<7} {'Rel.':<7} {'Memory'}")
print("-" * 80)

for i, (mem, scores) in enumerate(results2, 1):
    bar_len = int(scores['total'] * 50)
    bar = "█" * bar_len
    short_content = mem.content[:50] + "..." if len(mem.content) > 50 else mem.content
    print(f"  {i:<3} {scores['total']:<8.4f} {scores['recency']:<7.3f} {scores['importance']:<7.1f} {scores['relevance']:<7.2f} {short_content}")
    print(f"      {bar}")

print("\n" + "=" * 70)
print("สังเกต: ผลลัพธ์เปลี่ยนเพราะ relevance ต่างกัน!")
print("  - Query 1 (นม โปร) → ได้ memories เกี่ยวกับโปรโมชั่นนม")
print("  - Query 2 (บริการดี) → ได้ memories เกี่ยวกับประสบการณ์ที่ร้าน")
print("=" * 70)

## Part 5: ทดลองเอง! - เปลี่ยน Query ดู

ลองเปลี่ยน query ด้านล่าง แล้ว Run เพื่อดูผลลัพธ์

ลองค้นหา:
- `"ไข่ ซื้อ"` - หาเรื่องไข่
- `"ผงซักฟอก"` - หาเรื่องผงซักฟอก
- `"ครอบครัว ลูก"` - หาเรื่องครอบครัว
- `"CJ More ใกล้บ้าน"` - หาเรื่อง CJ More

In [ ]:
# ===================================
# ลองเปลี่ยน query ตรงนี้!
# ===================================
MY_QUERY = "ไข่ ซื้อ"  # ← เปลี่ยนตรงนี้!

print(f"Query: \"{MY_QUERY}\"")
print("-" * 60)

my_results = somsri.retrieve(MY_QUERY, top_k=5, current_time=NOW)

for i, (mem, scores) in enumerate(my_results, 1):
    bar = "█" * int(scores['total'] * 40)
    print(f"\n  #{i} Score: {scores['total']:.4f}")
    print(f"     Memory: \"{mem.content}\"")
    print(f"     Breakdown: recency={scores['recency']:.3f} × importance={scores['importance']:.1f} × relevance={scores['relevance']:.2f}")
    print(f"     {bar}")

## Part 6: Reflection - ข้อคิดจากประสบการณ์

**Reflection** คือ memory ชนิดพิเศษที่ agent สร้างขึ้นจากการ "คิดย้อน" เกี่ยวกับประสบการณ์ที่ผ่านมา

ตัวอย่าง:
- Observations: "ไป Big C คนเยอะ", "ต่อคิวนาน 20 นาที", "ของก็ไม่ได้ถูกมาก"
- **Reflection**: "Big C วันเสาร์คนเยอะมาก ควรไปวันธรรมดาแทน"

### ใน Paper จริง:
- Agent จะ reflect เมื่อ **ผลรวม importance** ของ observations ล่าสุดเกินค่า threshold
- LLM จะเป็นคนสร้าง reflection จาก observations เหล่านั้น
- Reflections จะถูกเก็บเข้า Memory Stream เหมือน memory ทั่วไป

มาดูว่า reflections ของสมศรีมีอะไรบ้าง:

In [ ]:
# ดู reflections ที่สมศรีมี
print("=" * 60)
print("Reflections ของ สมศรี (ข้อคิดจากประสบการณ์)")
print("=" * 60)

reflections = [m for m in somsri.memories if m.memory_type == "reflection"]

for i, ref in enumerate(reflections, 1):
    print(f"\n  Reflection #{i}:")
    print(f"    \"{ref.content}\"")
    print(f"    ความสำคัญ: {ref.importance}/10")
    if ref.related_store:
        print(f"    เกี่ยวกับร้าน: {ref.related_store}")

# แสดงว่า reflections เกิดจากอะไร
print("\n" + "-" * 60)
print("Reflections เกิดจากการรวมหลาย observations เข้าด้วยกัน:")
print()
print('  Observations: "Big C ของครบ" + "Big C คนเยอะ ต่อคิวนาน"')
print('       ↓ LLM คิด ↓')
print('  Reflection: "Big C มีของครบกว่า Lotus\'s โดยเฉพาะผงซักฟอก มีหลายยี่ห้อ"')
print()
print("  Reflection นี้จะถูกเก็บกลับเข้า Memory Stream")
print("  และจะถูกดึงมาใช้ตัดสินใจในอนาคต!")

## Part 7: สรุป

### สิ่งที่ได้เรียนรู้

1. **Memory Stream** เก็บทุกประสบการณ์ของ agent ในรูปภาษาธรรมชาติ

2. **สูตร Retrieval**: `score = recency × importance × relevance`
   - **Recency** (0.99^hours): ความทรงจำเก่าจะค่อยๆ จาง
   - **Importance** (1-10): LLM ให้คะแนนความสำคัญ
   - **Relevance** (keyword/vector): ตรงกับสิ่งที่กำลังคิดแค่ไหน

3. **เปลี่ยน query → เปลี่ยนผลลัพธ์** เพราะ relevance เปลี่ยน

4. **Reflection** = ข้อคิดจากประสบการณ์ที่ผ่านมา ถูกเก็บกลับเข้า Memory Stream

### Next Step
→ ไป **Notebook 02: Cognitive Loop** เพื่อดูว่า Memory Stream ถูกใช้ในการตัดสินใจอย่างไร!

In [ ]:
# ============================================================
# แบบฝึกหัด: ลองเพิ่ม memories ของตัวเอง!
# ============================================================

# 1. เพิ่ม memory ใหม่ 3 อัน (ลองคิดเหตุการณ์ใหม่ให้สมศรี)
# somsri.add("...", importance=?, memory_type="observation", created_at=NOW - timedelta(hours=?))

# 2. ลอง query ดูว่า memory ใหม่โผล่ขึ้นมาไหม
# results = somsri.retrieve("your query here", top_k=5, current_time=NOW)

# 3. ทดลอง: ถ้าเปลี่ยน RECENCY_DECAY จาก 0.99 เป็น 0.95 ผลจะต่างกันอย่างไร?
#    (0.95 = จำได้สั้นลง, 0.999 = จำได้นานขึ้น)

print("ลองทำแบบฝึกหัดข้างบนดูนะ!")
print("เปลี่ยน code แล้ว Run cell นี้ใหม่")